# Classification with new category discovery

Invoices keep coming in, one after another, and you want to sort each one into a category. But you can't know all the categories ahead of time. So let the categories grow as the invoices arrive: each invoice is one TypeLLM call that picks one of the categories you have, or `new`, and only then a second field names the new category.

1. Start with two categories.
2. Each invoice is sorted, and a new category is added when none fits.

Paste your TypeLLM API key in the cell below, then run all cells.

In [1]:
%pip install -q "typellm>=0.6.7"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Paste your API key here.
TYPELLM_API_KEY = ""  # https://typellm.ai/dashboard/keys

In [3]:
from typellm import TypeLLMClient

client = TypeLLMClient(api_key=TYPELLM_API_KEY)

INVOICES = [
    "AWS - EC2 and S3 usage, September - $1,842.17",
    "Slack Technologies - Pro plan, 12 seats, annual - $1,050.00",
    "Herman Miller - 4 Aeron office chairs - $5,560.00",
    "Staples - printer paper, toner and pens - $213.48",
    "Delta Air Lines - BOS to SFO, sales trip, J. Park - $612.40",
    "Hilton San Francisco - 3 nights, sales trip, J. Park - $1,137.00",
    "Nopa Restaurant - client dinner, 4 guests - $386.20",
    "Morgan & Lee LLP - legal retainer, October - $4,000.00",
    "Grant Thornton - annual financial audit - $18,500.00",
    "Google Ads - search campaign, September - $3,210.55",
    "Meta Platforms - Facebook ad campaign, September - $1,480.00",
    "Eversource - office electricity, September - $642.31",
    "Comcast Business - office internet, October - $249.99",
    "Seaport Properties - office rent, October - $12,400.00",
    "Ana Ruiz Design - logo and brand guide, contract work - $3,600.00",
    "Namecheap - domain renewal, 2 years - $31.96",
    "Blue Bottle Coffee - office coffee beans - $142.00",
    "SaaStr Annual - 2 conference passes - $1,998.00",
    "Hiscox - general liability insurance, annual premium - $2,145.00",
    "Apple - MacBook Pro 14-inch for new engineer - $2,399.00",
    "Uber - airport rides, sales trip, J. Park - $96.40",
    "Marriott Chicago - 2 nights, customer visit, L. Chen - $548.00",
    "United Airlines - SFO to ORD, customer visit, L. Chen - $431.20",
    "GitHub - Team plan, 15 seats, October - $60.00",
    "Figma - Professional plan, 6 editors, annual - $1,080.00",
    "Zoom Video Communications - Business plan, annual - $2,199.00",
    "Datadog - infrastructure monitoring, September - $912.00",
    "DoorDash - team lunch, 12 people - $274.85",
    "Sweetgreen - lunch with candidate - $48.30",
    "Office Depot - whiteboard markers and sticky notes - $67.12",
    "Dell - 3 UltraSharp 27-inch monitors - $1,349.97",
    "Logitech - 10 wireless keyboards and mice - $799.90",
    "LinkedIn - sponsored job post and ads, September - $1,250.00",
    "Printful - 100 branded t-shirts for conference booth - $1,420.00",
    "Verizon Business - 8 company phone lines, October - $520.00",
    "City Water Department - office water, Q3 - $118.64",
    "FedEx - overnight shipping of contracts - $84.50",
    "UPS - shipping laptops to remote hires - $212.30",
    "Gusto - payroll processing, October - $340.00",
    "Hired Talent Partners - placement fee, senior engineer - $27,000.00",
    "USPTO - trademark application, 2 classes - $700.00",
    "Delaware Division of Corporations - annual franchise tax - $450.00",
    "Chase - international wire fees, September - $135.00",
    "Blue Shield of California - employee health premiums, October - $9,860.00",
    "Coursera for Business - team learning licences, annual - $3,990.00",
    "O'Reilly Media - online learning, 5 seats - $2,495.00",
    "SparkleClean - office cleaning, October - $680.00",
    "Cavallo Point - team offsite venue, 2 days - $8,750.00",
    "Red Cross - company donation, disaster relief - $1,000.00",
    "Steelcase - 2 standing desks - $2,318.00",
]

## Start with two categories

They show the model how broad you want a category to be.

In [4]:
# Every call sees this list, and new categories are added to it.
categories = ["travel", "legal"]

## Classify and create categories in one API call

`category` chooses from the categories so far, plus `new`. `new_category` runs only when the answer is `new`, and names the category from what the invoice is. TypeLLM allows you to classify an invoice and create a new category in one API call with [`when`](https://typellm.ai/docs/workflows#conditional-fields). That is faster and cheaper, since TypeLLM only needs to read each invoice once.

In [5]:
def classify(invoice):
    """Returns (category, new category name or None)."""
    questions = {
        "category": {
            "type": "string", "enum": categories + ["new"], "thinking": "auto",
            "instructions": "Which category does this invoice belong to? Choose new unless one of them clearly fits.",
        },
        "new_category": {
            "type": "string", "when": {"category": "new"},
            "instructions": "A broad snake_case category name of one or two words, like travel or utilities, "
                            "that many other invoices could share.",
        },
    }
    result = client.generate(context=f"Invoice: {invoice}", questions=questions).result
    return result["category"], result.get("new_category")

In [6]:
assigned = {}  # invoice -> category
for invoice in INVOICES:
    category, new_name = classify(invoice)
    if category == "new":
        if new_name not in categories:
            categories.append(new_name)
        assigned[invoice] = new_name
        print(f"NEW  {new_name:22s} <- {invoice}")
    else:
        assigned[invoice] = category
        print(f"     {category:22s} <- {invoice}")

print(f"\n{len(categories)} categories from {len(INVOICES)} invoices")

NEW  cloud                  <- AWS - EC2 and S3 usage, September - $1,842.17


NEW  software               <- Slack Technologies - Pro plan, 12 seats, annual - $1,050.00


NEW  furniture              <- Herman Miller - 4 Aeron office chairs - $5,560.00


NEW  office_supplies        <- Staples - printer paper, toner and pens - $213.48


     travel                 <- Delta Air Lines - BOS to SFO, sales trip, J. Park - $612.40


...

     software               <- O'Reilly Media - online learning, 5 seats - $2,495.00


     professional_services  <- SparkleClean - office cleaning, October - $680.00


     conferences            <- Cavallo Point - team offsite venue, 2 days - $8,750.00


NEW  donations              <- Red Cross - company donation, disaster relief - $1,000.00


     furniture              <- Steelcase - 2 standing desks - $2,318.00

18 categories from 50 invoices


In [7]:
for name in categories:
    members = [invoice.split(" - ")[0] for invoice, c in assigned.items() if c == name]
    print(f"{name:22s} {len(members)}: {', '.join(members)}")

travel                 5: Delta Air Lines, Hilton San Francisco, Uber, Marriott Chicago, United Airlines
legal                  2: Morgan & Lee LLP, USPTO
cloud                  2: AWS, Namecheap
software               8: Slack Technologies, GitHub, Figma, Zoom Video Communications, Datadog, Gusto, Coursera for Business, O'Reilly Media
furniture              2: Herman Miller, Steelcase
office_supplies        3: Staples, Blue Bottle Coffee, Office Depot
meals                  3: Nopa Restaurant, DoorDash, Sweetgreen
professional_services  4: Grant Thornton, Ana Ruiz Design, Hired Talent Partners, SparkleClean
marketing              3: Google Ads, Meta Platforms, LinkedIn
utilities              4: Eversource, Comcast Business, Verizon Business, City Water Department
rent                   1: Seaport Properties
conferences            3: SaaStr Annual, Printful, Cavallo Point
insurance              2: Hiscox, Blue Shield of California
equipment              3: Apple, Dell, Logitech
shippin